In [ ]:

# ====================
# 1. KONFIGURASI & SETUP
# ====================

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import seaborn as sns
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

# Setup plotting
plt.style.use('default')
sns.set_style("whitegrid")

@dataclass
class CookingConfig:
    """Konfigurasi parameter memasak"""
    # Parameter bahan
    rice_mass: float = 10.0          # kg
    water_mass: float = 15.0         # kg
    pan_mass: float = 5.0           # kg
    
    # Parameter termal
    initial_temp: float = 25.0      # °C
    target_temp: float = 100.0      # °C
    ambient_temp: float = 25.0      # °C
    boiling_temp: float = 100.0     # °C
    
    # Parameter proses
    gelatinization_start: float = 60.0   # °C
    gelatinization_end: float = 80.0     # °C
    gelatinization_time: float = 15.0    # menit
    
    # Parameter peralatan
    burner_power: float = 3000.0    # Watt
    pan_surface_area: float = 0.5   # m²
    
    # Koefisien fisik
    Cp_water: float = 4186.0        # J/kg°C
    Cp_rice: float = 2000.0         # J/kg°C
    Cp_pan: float = 500.0           # J/kg°C
    latent_heat_vaporization: float = 2260000.0  # J/kg
    
    # Koefisien transfer panas
    heat_transfer_coeff: float = 50.0    # W/m²°C
    ambient_loss_coeff: float = 10.0     # W/m²°C
    heating_efficiency: float = 0.7      # efisiensi
    
    # Parameter simulasi
    simulation_time: float = 60.0    # menit
    time_step: float = 1.0          # detik
    
    # Atribut yang dihitung (bukan parameter input)
    total_mass: float = field(init=False, default=None)
    
    def __post_init__(self):
        """Validasi konfigurasi dan hitung atribut turunan"""
        self.total_mass = self.rice_mass + self.water_mass
        if self.water_mass / self.rice_mass < 1.0:
            print("Peringatan: Rasio air:beras terlalu rendah (< 1:1)")
    
    def copy(self):
        """Buat salinan konfigurasi"""
        # Hanya copy parameter input, bukan atribut yang dihitung
        params = {k: v for k, v in self.__dict__.items() 
                 if k not in ['total_mass']}
        # Buat instance baru
        new_config = CookingConfig(**params)
        return new_config
    
    def update_parameter(self, parameter_name: str, value: float):
        """Update satu parameter dan hitung ulang atribut turunan"""
        if parameter_name in self.__annotations__:
            setattr(self, parameter_name, value)
            # Hitung ulang atribut turunan
            self.__post_init__()
        else:
            raise ValueError(f"Parameter {parameter_name} tidak valid")

print("KONFIGURASI & SETUP")

In [ ]:
# ====================
# 2. MODEL FISIKA
# ====================

class PhysicsModel:
    """Model fisika untuk proses memasak"""
    
    def __init__(self, config: CookingConfig):
        self.config = config
    
    def calculate_effective_heat_capacity(self) -> float:
        """Hitung kapasitas panas efektif sistem"""
        # Mass fractions
        water_frac = self.config.water_mass / self.config.total_mass
        rice_frac = self.config.rice_mass / self.config.total_mass
        
        # Heat capacity of mixture
        Cp_mix = (water_frac * self.config.Cp_water + 
                 rice_frac * self.config.Cp_rice)
        
        # Total heat capacity including pan
        total_capacity = (self.config.total_mass * Cp_mix + 
                         self.config.pan_mass * self.config.Cp_pan)
        
        return total_capacity
    
    def heat_input(self, temperature: float, water_content: float) -> float:
        """Hitung input panas dari burner"""
        if (temperature < self.config.target_temp and 
            water_content > 0.1 * self.config.water_mass):
            return self.config.burner_power * self.config.heating_efficiency
        return 0.0
    
    def heat_loss(self, temperature: float) -> float:
        """Hitung kehilangan panas ke lingkungan"""
        delta_T = temperature - self.config.ambient_temp
        return (self.config.ambient_loss_coeff * 
                self.config.pan_surface_area * delta_T)
    
    def evaporation_rate(self, temperature: float) -> float:
        """Hitung laju penguapan"""
        if temperature >= self.config.boiling_temp:
            # Evaporation increases with temperature above boiling
            rate_base = 0.01  # kg/menit
            temp_factor = (temperature - self.config.boiling_temp) / 10.0 + 1.0
            return rate_base * temp_factor / 60.0  # konversi ke kg/detik
        return 0.0
    
    def gelatinization_rate(self, temperature: float) -> float:
        """Hitung laju gelatinisasi"""
        if (temperature >= self.config.gelatinization_start and 
            temperature <= self.config.gelatinization_end):
            # Linear rate based on temperature
            temp_range = self.config.gelatinization_end - self.config.gelatinization_start
            return 0.1 * (temperature - self.config.gelatinization_start) / temp_range
        return 0.0

print("MODEL FISIKA")

In [ ]:
# ====================
# 3. SISTEM PERSAMAAN DIFERENSIAL
# ====================

class DifferentialEquations:
    """Sistem persamaan diferensial untuk simulasi kontinu"""
    
    def __init__(self, physics_model: PhysicsModel):
        self.physics = physics_model
        self.config = physics_model.config
    
    def system_equations(self, t: float, y: np.ndarray) -> np.ndarray:
        """
        Sistem persamaan diferensial:
        y = [temperature, water_mass, gelatinization]
        
        Returns:
        dy/dt = [dT/dt, dm/dt, dG/dt]
        """
        T, m_water, G = y
        
        # 1. Effective heat capacity
        effective_cp = self.physics.calculate_effective_heat_capacity()
        
        # 2. Heat terms
        Q_in = self.physics.heat_input(T, m_water)
        Q_loss = self.physics.heat_loss(T)
        
        # 3. Evaporation
        evap_rate = self.physics.evaporation_rate(T)
        dm_dt = -evap_rate if m_water > 0 else 0.0
        Q_evap = evap_rate * self.config.latent_heat_vaporization if m_water > 0 else 0.0
        
        # 4. Gelatinization
        gel_rate = self.physics.gelatinization_rate(T)
        dG_dt = gel_rate if G < 1.0 else 0.0
        Q_gel = 100.0 * dG_dt  # Heat for chemical process
        
        # 5. Temperature change
        net_heat = Q_in - Q_loss - Q_evap - Q_gel
        dT_dt = net_heat / effective_cp if effective_cp > 0 else 0.0
        
        # Limit temperature change rate
        dT_dt = np.clip(dT_dt, -0.5, 2.0)
        
        return np.array([dT_dt, dm_dt, dG_dt])
    
    def get_initial_conditions(self) -> np.ndarray:
        """Kondisi awal sistem"""
        return np.array([
            self.config.initial_temp,      # temperature
            self.config.water_mass,        # water mass
            0.0                            # gelatinization progress
        ])

print("SISTEM PERSAMAAN DIFERENSIAL")

In [ ]:
# ====================
# 4. SIMULATOR UTAMA
# ====================

class RiceCookingSimulator:
    """Simulator utama proses memasak nasi"""
    
    def __init__(self, config: CookingConfig):
        self.config = config
        self.physics = PhysicsModel(config)
        self.equations = DifferentialEquations(self.physics)
        
        # Results storage
        self.time_history = None
        self.temperature_history = None
        self.water_history = None
        self.gelatinization_history = None
        self.results = None
    
    def run_simulation(self) -> Dict:
        """Jalankan simulasi"""
        print("Memulai simulasi proses memasak...")
        
        # Setup time
        t_span = (0, self.config.simulation_time * 60)  # Convert to seconds
        t_eval = np.arange(0, self.config.simulation_time * 60, 
                          self.config.time_step)
        
        # Initial conditions
        y0 = self.equations.get_initial_conditions()
        
        # Solve ODE system
        solution = solve_ivp(
            fun=self.equations.system_equations,
            t_span=t_span,
            y0=y0,
            t_eval=t_eval,
            method='RK45',
            rtol=1e-6,
            atol=1e-9,
            dense_output=True
        )
        
        # Store results
        self.time_history = solution.t / 60.0  # Convert to minutes
        self.temperature_history = solution.y[0]
        self.water_history = solution.y[1] / self.config.water_mass * 100.0
        self.gelatinization_history = solution.y[2] * 100.0
        
        # Calculate metrics
        self.results = self._calculate_metrics()
        
        print("Simulasi selesai!")
        return self.results
    
    def _calculate_metrics(self) -> Dict:
        """Hitung metrik kualitas memasak"""
        if self.time_history is None:
            raise ValueError("Jalankan simulasi terlebih dahulu")
        
        metrics = {
            # Time metrics
            'time_to_boiling': self._get_time_to_temperature(self.config.boiling_temp),
            'time_to_target': self._get_time_to_temperature(self.config.target_temp),
            'time_full_gelatinization': self._get_time_to_gelatinization(95.0),
            
            # Temperature metrics
            'max_temperature': np.max(self.temperature_history),
            'avg_temperature': np.mean(self.temperature_history),
            'final_temperature': self.temperature_history[-1],
            
            # Quality metrics
            'final_gelatinization': self.gelatinization_history[-1],
            'final_water_content': self.water_history[-1],
            'max_water_loss': 100.0 - np.min(self.water_history),
            
            # Energy metrics
            'energy_consumption': self._calculate_energy_consumption(),
            'cooking_efficiency': self._calculate_efficiency()
        }
        
        return metrics
    
    def _get_time_to_temperature(self, target_temp: float) -> Optional[float]:
        """Waktu untuk mencapai suhu tertentu"""
        indices = np.where(self.temperature_history >= target_temp)[0]
        if len(indices) > 0:
            return self.time_history[indices[0]]
        return None
    
    def _get_time_to_gelatinization(self, target_gel: float) -> Optional[float]:
        """Waktu untuk mencapai tingkat gelatinisasi tertentu"""
        indices = np.where(self.gelatinization_history >= target_gel)[0]
        if len(indices) > 0:
            return self.time_history[indices[0]]
        return None
    
    def _calculate_energy_consumption(self) -> float:
        """Hitung konsumsi energi total"""
        cooking_time = self.time_history[-1] * 60.0  # seconds
        energy = self.config.burner_power * cooking_time / 3600000.0  # kWh
        return energy
    
    def _calculate_efficiency(self) -> float:
        """Hitung efisiensi memasak"""
        # Heat required to cook rice (theoretical minimum)
        heat_required = (self.config.total_mass * self.physics.calculate_effective_heat_capacity() *
                        (self.config.target_temp - self.config.initial_temp))
        
        # Actual energy used
        energy_used = self._calculate_energy_consumption() * 3600000.0  # Joules
        
        if energy_used > 0:
            return min(heat_required / energy_used * 100.0, 100.0)
        return 0.0

print("SIMULATOR UTAMA")

In [ ]:

# ====================
# 5. VISUALISASI
# ====================

class Visualization:
    """Kelas untuk visualisasi hasil simulasi"""
    
    @staticmethod
    def plot_temperature_profile(simulator: RiceCookingSimulator, 
                                ax: plt.Axes = None):
        """Plot profil suhu"""
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 6))
        
        time = simulator.time_history
        temp = simulator.temperature_history
        config = simulator.config
        
        ax.plot(time, temp, 'b-', linewidth=2.5, label='Suhu Nasi', alpha=0.8)
        
        # Reference lines
        ax.axhline(y=config.target_temp, color='r', linestyle='--', 
                  linewidth=1.5, alpha=0.7, label=f'Target ({config.target_temp}°C)')
        ax.axhline(y=config.boiling_temp, color='g', linestyle='--', 
                  linewidth=1.5, alpha=0.7, label=f'Titik Didih ({config.boiling_temp}°C)')
        ax.axhline(y=config.gelatinization_start, color='orange', linestyle=':', 
                  linewidth=1.5, alpha=0.6, label=f'Mulai Gelatinisasi')
        ax.axhline(y=config.gelatinization_end, color='orange', linestyle=':', 
                  linewidth=1.5, alpha=0.6, label=f'Akhir Gelatinisasi')
        
        # Gelatinization zone
        ax.fill_between(time, config.gelatinization_start, config.gelatinization_end,
                       alpha=0.1, color='orange', label='Zona Gelatinisasi Optimal')
        
        ax.set_xlabel('Waktu (menit)', fontsize=12)
        ax.set_ylabel('Suhu (°C)', fontsize=12)
        ax.set_title('Profil Suhu Selama Memasak', fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        return ax
    
    @staticmethod
    def plot_quality_metrics(simulator: RiceCookingSimulator):
        """Plot metrik kualitas"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        time = simulator.time_history
        
        # Plot 1: Gelatinization progress
        ax1 = axes[0, 0]
        ax1.plot(time, simulator.gelatinization_history, 'g-', linewidth=2.5)
        ax1.axhline(y=95, color='r', linestyle='--', alpha=0.7, label='95% (Matang)')
        ax1.set_xlabel('Waktu (menit)')
        ax1.set_ylabel('Gelatinisasi (%)')
        ax1.set_title('Proses Gelatinisasi Pati')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim([0, 105])
        
        # Plot 2: Water content
        ax2 = axes[0, 1]
        ax2.plot(time, simulator.water_history, 'purple', linewidth=2.5)
        ax2.axhline(y=50, color='r', linestyle='--', alpha=0.7, label='Batas Minimal 50%')
        ax2.set_xlabel('Waktu (menit)')
        ax2.set_ylabel('Kandungan Air (%)')
        ax2.set_title('Perubahan Kandungan Air')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Phase diagram
        ax3 = axes[1, 0]
        Visualization._plot_phase_diagram(simulator, ax3)
        
        # Plot 4: Energy accumulation
        ax4 = axes[1, 1]
        Visualization._plot_energy_accumulation(simulator, ax4)
        
        plt.tight_layout()
        return fig
    
    @staticmethod
    def _plot_phase_diagram(simulator: RiceCookingSimulator, ax: plt.Axes):
        """Plot diagram fase proses"""
        time = simulator.time_history
        temp = simulator.temperature_history
        config = simulator.config
        
        # Define phases
        phases = []
        for T in temp:
            if T < config.gelatinization_start:
                phases.append(0)  # Pemanasan awal
            elif T < config.gelatinization_end:
                phases.append(1)  # Gelatinisasi
            elif T < config.boiling_temp:
                phases.append(2)  # Pemanasan lanjut
            elif T >= config.boiling_temp:
                phases.append(3)  # Pendidihan
            else:
                phases.append(4)  # Pendinginan
        
        # Color mapping
        phase_colors = ['lightblue', 'lightgreen', 'yellow', 'orange', 'red']
        phase_labels = ['Pemanasan Awal', 'Gelatinisasi', 'Pemanasan Lanjut', 
                       'Pendidihan', 'Pematangan']
        
        # Plot phases
        current_phase = phases[0]
        start_idx = 0
        
        for i, phase in enumerate(phases[1:], 1):
            if phase != current_phase or i == len(phases) - 1:
                ax.fill_betweenx([0, 100], time[start_idx], time[i-1],
                               color=phase_colors[current_phase], alpha=0.4)
                mid_time = (time[start_idx] + time[i-1]) / 2
                ax.text(mid_time, 50, phase_labels[current_phase], 
                       ha='center', va='center', rotation=90,
                       fontweight='bold', fontsize=9)
                current_phase = phase
                start_idx = i
        
        ax.set_xlabel('Waktu (menit)')
        ax.set_ylabel('Intensitas Proses')
        ax.set_title('Diagram Fase Memasak')
        ax.set_ylim([0, 100])
        ax.grid(True, alpha=0.3)
    
    @staticmethod
    def _plot_energy_accumulation(simulator: RiceCookingSimulator, ax: plt.Axes):
        """Plot akumulasi energi"""
        time = simulator.time_history
        temp = simulator.temperature_history
        config = simulator.config
        
        # Calculate cumulative energy
        energy = np.zeros_like(time)
        for i in range(1, len(time)):
            dt = (time[i] - time[i-1]) * 60  # seconds
            if temp[i] < config.target_temp:
                energy[i] = energy[i-1] + config.burner_power * dt / 3600000  # kWh
            else:
                energy[i] = energy[i-1]
        
        ax.plot(time, energy, 'b-', linewidth=2, label='Energi Kumulatif')
        ax.set_xlabel('Waktu (menit)')
        ax.set_ylabel('Energi (kWh)')
        ax.set_title('Akumulasi Konsumsi Energi')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Add efficiency annotation
        efficiency = simulator.results['cooking_efficiency']
        ax.text(0.05, 0.95, f'Efisiensi: {efficiency:.1f}%', 
               transform=ax.transAxes, fontsize=10,
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    @staticmethod
    def plot_comparison_chart(simulators: List[RiceCookingSimulator], 
                             labels: List[str]):
        """Plot perbandingan beberapa simulasi"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        colors = plt.cm.tab10(np.linspace(0, 1, len(simulators)))
        
        # Plot 1: Temperature comparison
        ax1 = axes[0, 0]
        for i, sim in enumerate(simulators):
            ax1.plot(sim.time_history, sim.temperature_history, 
                    color=colors[i], linewidth=2, label=labels[i], alpha=0.8)
        ax1.set_xlabel('Waktu (menit)')
        ax1.set_ylabel('Suhu (°C)')
        ax1.set_title('Perbandingan Profil Suhu')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Gelatinization comparison
        ax2 = axes[0, 1]
        for i, sim in enumerate(simulators):
            ax2.plot(sim.time_history, sim.gelatinization_history,
                    color=colors[i], linewidth=2, label=labels[i], alpha=0.8)
        ax2.set_xlabel('Waktu (menit)')
        ax2.set_ylabel('Gelatinisasi (%)')
        ax2.set_title('Perbandingan Proses Gelatinisasi')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Plot 3: Water content comparison
        ax3 = axes[1, 0]
        for i, sim in enumerate(simulators):
            ax3.plot(sim.time_history, sim.water_history,
                    color=colors[i], linewidth=2, label=labels[i], alpha=0.8)
        ax3.set_xlabel('Waktu (menit)')
        ax3.set_ylabel('Kandungan Air (%)')
        ax3.set_title('Perbandingan Kandungan Air')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        # Plot 4: Metrics comparison (bar chart)
        ax4 = axes[1, 1]
        metrics = ['time_to_target', 'final_gelatinization', 
                  'final_water_content', 'energy_consumption']
        metric_labels = ['Waktu ke 100°C (mnt)', 'Gelatinisasi Akhir (%)',
                        'Air Akhir (%)', 'Energi (kWh)']
        
        x = np.arange(len(metrics))
        width = 0.8 / len(simulators)
        
        for i, sim in enumerate(simulators):
            values = [sim.results[metric] for metric in metrics]
            # Handle None values
            values = [0 if v is None else v for v in values]
            offset = (i - len(simulators)/2 + 0.5) * width
            ax4.bar(x + offset, values, width, label=labels[i], 
                   color=colors[i], alpha=0.7)
        
        ax4.set_xlabel('Metrik')
        ax4.set_ylabel('Nilai')
        ax4.set_title('Perbandingan Metrik Kinerja')
        ax4.set_xticks(x)
        ax4.set_xticklabels(metric_labels, rotation=45, ha='right')
        ax4.legend()
        ax4.grid(True, alpha=0.3, axis='y')
        
        plt.tight_layout()
        return fig

print("VISUALISASI")

In [ ]:

# ====================
# 6. ANALISIS SENSITIVITAS
# ====================

class SensitivityAnalysis:
    """Analisis sensitivitas parameter"""
    
    @staticmethod
    def analyze_parameter_sensitivity(base_config: CookingConfig,
                                     parameter_name: str,
                                     values: List[float]) -> Dict:
        """
        Analisis sensitivitas untuk satu parameter
        
        Returns:
            Dictionary dengan hasil untuk setiap nilai parameter
        """
        results = []
        
        for value in values:
            # Create new config with modified parameter menggunakan copy()
            config = base_config.copy()
            config.update_parameter(parameter_name, value)
            
            # Run simulation
            simulator = RiceCookingSimulator(config)
            metrics = simulator.run_simulation()
            
            results.append({
                'value': value,
                'simulator': simulator,
                'metrics': metrics
            })
        
        return {
            'parameter': parameter_name,
            'results': results
        }
    
    @staticmethod
    def multi_parameter_analysis(base_config: CookingConfig,
                               parameter_ranges: Dict[str, List[float]]) -> Dict:
        """
        Analisis sensitivitas untuk multiple parameters
        """
        all_results = {}
        
        for param_name, values in parameter_ranges.items():
            print(f"Analisis sensitivitas untuk parameter: {param_name}")
            results = SensitivityAnalysis.analyze_parameter_sensitivity(
                base_config, param_name, values
            )
            all_results[param_name] = results
        
        return all_results
    
    @staticmethod
    def plot_sensitivity_results(analysis_results: Dict, 
                                metric_name: str = 'time_to_target'):
        """Plot hasil analisis sensitivitas"""
        parameter = analysis_results['parameter']
        results = analysis_results['results']
        
        values = [r['value'] for r in results]
        metrics = []
        
        # Handle None values
        for r in results:
            metric_value = r['metrics'][metric_name]
            if metric_value is None:
                metrics.append(0)  # atau np.nan
            else:
                metrics.append(metric_value)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.plot(values, metrics, 'o-', linewidth=2, markersize=8)
        ax.set_xlabel(f'Nilai {parameter}', fontsize=12)
        ax.set_ylabel(metric_name.replace('_', ' ').title(), fontsize=12)
        ax.set_title(f'Sensitivitas: {parameter} vs {metric_name}', 
                    fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Add annotations
        for i, (x, y) in enumerate(zip(values, metrics)):
            if y > 0:  # Skip zero values (None)
                ax.annotate(f'{y:.1f}', (x, y), textcoords="offset points",
                           xytext=(0,10), ha='center', fontsize=9)
        
        return fig, ax

print("ANALISIS SENSITIVITAS")

In [ ]:
# ====================
# 7. Menjalankan Simulasi
# ====================

def main():
    """Program utama"""
    print("="*60)
    print("SIMULASI KONTINU PROSES MEMASAK NASI")
    print("="*60)
    
    # 1. Setup konfigurasi
    print("\n1. Setup Konfigurasi...")
    config = CookingConfig(
        rice_mass=10.0,
        water_mass=15.0,
        initial_temp=25.0,
        target_temp=100.0,
        burner_power=3000.0,
        simulation_time=60.0
    )
    
    # 2. Jalankan simulasi utama
    print("\n2. Menjalankan Simulasi Utama...")
    simulator = RiceCookingSimulator(config)
    results = simulator.run_simulation()
    
    # 3. Tampilkan hasil
    print("\n3. Hasil Simulasi:")
    print("-"*40)
    for key, value in results.items():
        if value is not None:
            if isinstance(value, float):
                print(f"{key:30}: {value:.2f}")
            else:
                print(f"{key:30}: {value}")
    
    # 4. Visualisasi
    print("\n4. Membuat Visualisasi...")
    
    # Plot utama
    fig1, ax1 = plt.subplots(figsize=(12, 7))
    Visualization.plot_temperature_profile(simulator, ax1)
    plt.show()
    
    # Plot metrik kualitas
    Visualization.plot_quality_metrics(simulator)
    plt.show()
    
    # 5. Analisis sensitivitas
    print("\n5. Analisis Sensitivitas...")
    
    # Analisis sensitivitas daya burner
    burner_power_values = [2000, 2500, 3000, 3500, 4000]
    burner_analysis = SensitivityAnalysis.analyze_parameter_sensitivity(
        config, 'burner_power', burner_power_values
    )
    
    fig2, ax2 = SensitivityAnalysis.plot_sensitivity_results(
        burner_analysis, 'time_to_target'
    )
    ax2.set_title('Sensitivitas Daya Burner vs Waktu Memasak')
    plt.show()
    
    # Analisis sensitivitas rasio air
    water_values = [12, 13, 14, 15, 16, 17, 18]
    water_analysis = SensitivityAnalysis.analyze_parameter_sensitivity(
        config, 'water_mass', water_values
    )
    
    fig3, ax3 = SensitivityAnalysis.plot_sensitivity_results(
        water_analysis, 'final_water_content'
    )
    ax3.set_title('Sensitivitas Jumlah Air vs Kandungan Air Akhir')
    plt.show()
    
    # 6. Simulasi perbandingan
    print("\n6. Simulasi Perbandingan Skenario...")
    
    # Buat beberapa skenario
    scenarios = [
        ("Daya Rendah (2500W)", CookingConfig(burner_power=2500)),
        ("Standar (3000W)", config),
        ("Daya Tinggi (3500W)", CookingConfig(burner_power=3500)),
        ("Air Sedikit (12kg)", CookingConfig(water_mass=12)),
        ("Air Banyak (18kg)", CookingConfig(water_mass=18)),
    ]
    
    simulators = []
    labels = []
    
    for label, scenario_config in scenarios:
        sim = RiceCookingSimulator(scenario_config)
        sim.run_simulation()
        simulators.append(sim)
        labels.append(label)
    
    # Plot perbandingan
    Visualization.plot_comparison_chart(simulators, labels)
    plt.show()
    
    print("\n" + "="*60)
    print("SIMULASI SELESAI")
    print("="*60)

def contoh_penggunaan():
    """Contoh penggunaan modular"""
    print("\nCONTOH PENGGUNAAN MODULAR:")
    print("-"*40)
    
    # Contoh 1: Simulasi cepat
    config_simple = CookingConfig(simulation_time=30)
    simulator = RiceCookingSimulator(config_simple)
    results = simulator.run_simulation()
    
    print(f"Waktu ke 100°C: {results['time_to_target']:.1f} menit")
    print(f"Gelatinisasi akhir: {results['final_gelatinization']:.1f}%")
    
    # Contoh 2: Visualisasi cepat
    Visualization.plot_temperature_profile(simulator)
    plt.title("Contoh Simulasi Cepat")
    plt.show()
    
    # Contoh 3: Analisis parameter tunggal
    analysis = SensitivityAnalysis.analyze_parameter_sensitivity(
        config_simple,
        'burner_power',
        [2000, 3000, 4000]
    )
    
    print("\nAnalisis Sensitivitas Daya Burner:")
    for result in analysis['results']:
        print(f"Daya {result['value']}W: {result['metrics']['time_to_target']:.1f} menit")

# Jalankan program
if __name__ == "__main__":
    main()